# Desafio

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Versao com loops (distancia euclidiana)
class KNNLoops:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        previsoes = []
        for x in X:
            dists = np.zeros(len(self.X_train))
            for j in range(len(self.X_train)):
                diff = self.X_train[j] - x
                dists[j] = np.sqrt(np.sum(diff ** 2))
            indices = np.argsort(dists, kind="stable")[:self.k]
            votos = np.bincount(self.y_train[indices])
            previsoes.append(votos.argmax())
        return np.array(previsoes)

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

# Versao vetorizada com broadcasting (distancia de manhattan)
class KNNBroadcasting:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.classes, self.y_cod = np.unique(np.asarray(y), return_inverse=True)
        return self

    def _distancias(self, X):
        dif = X[:, None, :] - self.X_train[None, :, :]
        return np.sum(np.abs(dif), axis=2)

    def predict(self, X):
        d = self._distancias(np.asarray(X, dtype=float))
        indices = np.argsort(d, axis=1, kind="stable")[:, :self.k]
        viz = self.y_cod[indices]
        venc = [np.bincount(v).argmax() for v in viz]
        return self.classes[venc]

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))


In [10]:
# Dados sinteticos com escalas diferentes (Idade x Renda)
rng = np.random.default_rng(42)
n = 120
idade = rng.uniform(20, 70, size=n)
renda = rng.uniform(20000, 150000, size=n)

# Regra dependente de ambas as caracteristicas
y = np.logical_or((idade > 45) & (renda > 60000), renda > 105000).astype(int)
X = np.column_stack([idade, renda])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Treino:", X_train.shape, "Teste:", X_test.shape)


Treino: (90, 2) Teste: (30, 2)


In [11]:
m_loop = KNNLoops(k=3).fit(X_train, y_train)
m_broad = KNNBroadcasting(k=3).fit(X_train, y_train)

pred_loop = m_loop.predict(X_test)
pred_broad = m_broad.predict(X_test)

sk_euc = KNeighborsClassifier(n_neighbors=3, metric="euclidean").fit(X_train, y_train)
print("KNNLoops bate com sklearn euclidiana?", np.array_equal(pred_loop, sk_euc.predict(X_test)))

sk_man = KNeighborsClassifier(n_neighbors=3, metric="manhattan").fit(X_train, y_train)
print("KNNBroadcasting bate com sklearn manhattan?", np.array_equal(pred_broad, sk_man.predict(X_test)))


KNNLoops bate com sklearn euclidiana? True
KNNBroadcasting bate com sklearn manhattan? True


In [12]:
score_sem_esc = m_broad.score(X_test, y_test)

# Ajuste do scaler  sobre o treino
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)

m_com_esc = KNNBroadcasting(k=3).fit(X_train_esc, y_train)
score_com_esc = m_com_esc.score(X_test_esc, y_test)

print("Acuracia sem escalonamento:", round(score_sem_esc, 4))
print("Acuracia com escalonamento:", round(score_com_esc, 4))


Acuracia sem escalonamento: 0.8
Acuracia com escalonamento: 0.9


In [13]:
# Medicao de tempo com perf_counter - busca por fora
ini = perf_counter()
m_loop.predict(X_test)
tempo_loops = perf_counter() - ini

ini = perf_counter()
m_broad.predict(X_test)
tempo_broad = perf_counter() - ini

print("Tempo Loops - Euclidiana:", f"{tempo_loops:.6f} s")
print("Tempo Broadcasting - Manhattan:", f"{tempo_broad:.6f} s")
print("Broadcasting foi", round(tempo_loops / tempo_broad, 1), "vezes mais rapido que Loops")


Tempo Loops - Euclidiana: 0.008944 s
Tempo Broadcasting - Manhattan: 0.000564 s
Broadcasting foi 15.8 vezes mais rapido que Loops


### Conclusão dos Resultados

- Implementações: A classe KNNLoops utiliza exclusivamente a distância Euclidiana via laços explícitos, enquanto a classe KNNBroadcasting utiliza exclusivamente a distância de Manhattan de forma vetorizada.
- Validação com Sklearn: Cada implementação foi comparada e bateu com o KNeighborsClassifier do scikit-learn nas respectivas configurações de métrica.
- Desempenho: O Broadcasting em Manhattan foi mais rápido que os loops em Euclidiana.